# 2. Reproduce the tobacco detector experiments

Train YOLOv8n/m/x on the frozen split, then evaluate selected checkpoints on validation and test images. All scientific code is defined below. Default execution explains the workflow without loading YOLO, downloading weights, or training. Install `requirements-experiments.txt` before opting in.


In [ ]:
from pathlib import Path
import sys
import os
import json
import tempfile

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'notebooks').is_dir() and (p / 'experiments/splits/items.csv').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from the repository root or a subdirectory.')
if str(ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(ROOT / 'notebooks'))
from reproduction import (new_run, frozen_assignments, write_local_splits,
                          check_detector_results, capture_environment)
os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'tobacco-notebook-mpl'))
os.environ.setdefault('YOLO_CONFIG_DIR', str(ROOT / 'experiments/.ultralytics'))
import pandas as pd
from IPython.display import display
RESULTS = ROOT / 'experiments/results'


## Choose inputs and a new output directory

Provide the original dataset in this layout:

```text
tobacco_yolo/
  images/<image-name>.jpg
  labels/<same-image-name>.txt
```

Each label row is `class_id x_center y_center width height`, with normalized coordinates. **Class 0 must mean tobacco**, not weeds. Software can check IDs but cannot determine their semantic meaning; confirm it from the annotation source.

Every new run uses a fresh directory under `experiments/reproductions/`. Reusing a run name raises an error to protect previous results. To retry a failed experiment, choose another name. No historical result is overwritten.


In [ ]:
RUN_TRAINING = False
CLASS_ZERO_CONFIRMED_TOBACCO = False
DATASET = ROOT / 'data/tobacco_yolo'
RUN_NAME = 'detector-reproduction-01'
MODELS = ['yolov8n.pt', 'yolov8m.pt', 'yolov8x.pt']
TRAINING_OPTIONS = dict(
    epochs=50, imgsz=1024, batch=14, device='0', optimizer='SGD', lr0=0.005,
    momentum=0.937, weight_decay=0.0005, cos_lr=True, amp=True,
)
run_dir = None
training_complete = False
print('Training enabled:', RUN_TRAINING)
display(pd.Series(TRAINING_OPTIONS, name='New run settings'))


In [ ]:
import csv
import re
import shutil
import time
from collections import Counter
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}


## Read YOLO annotations

Each nonempty row contains class ID and four normalized box coordinates. Class 0 must mean tobacco; software cannot verify that semantic claim.


In [ ]:
def image_paths(image_dir: Path) -> list[Path]:
    paths = sorted(path for path in image_dir.iterdir()
                   if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    stems = {}
    for path in paths:
        if path.stem in stems:
            raise ValueError(f"Duplicate image stem: {stems[path.stem]} and {path}")
        stems[path.stem] = path
    return paths

def parse_label(path: Path) -> tuple[list[int], list[str]]:
    class_ids: list[int] = []
    errors: list[str] = []
    if not path.exists():
        return class_ids, errors
    for line_number, raw_line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            errors.append(f"{path}:{line_number}: expected 5 columns, found {len(parts)}")
            continue
        try:
            class_id = int(parts[0])
            values = [float(value) for value in parts[1:]]
        except ValueError:
            errors.append(f"{path}:{line_number}: non-numeric YOLO value")
            continue
        if class_id < 0:
            errors.append(f"{path}:{line_number}: negative class id {class_id}")
        for value in values:
            if not 0 <= value <= 1:
                errors.append(f"{path}:{line_number}: normalized coordinate outside [0, 1]: {value}")
        class_ids.append(class_id)
    return class_ids, errors


### Try the annotation parser

This temporary synthetic label illustrates the file format without claiming dataset evidence.


In [ ]:
with tempfile.TemporaryDirectory(prefix='tobacco-label-example-') as example_dir:
    example_label = Path(example_dir) / 'example.txt'
    example_label.write_text('0 0.5 0.5 0.2 0.2\n', encoding='utf-8')
    example_classes, example_errors = parse_label(example_label)
display(pd.Series({'class_ids': example_classes, 'errors': example_errors}))


## Audit the dataset

Count images, labels, boxes, malformed rows, and filename groups. Prefix groups are a leakage diagnostic, not proof of independent acquisition.


In [ ]:
def infer_prefixes(stems: list[str]) -> dict[str, int]:
    patterns = [
        re.compile(r"^(.*?)(?:[_-]?frame[_-]?\d+)$", re.IGNORECASE),
        re.compile(r"^(.*?)(?:[_-]?\d{3,})$"),
    ]
    counts: Counter[str] = Counter()
    for stem in stems:
        for pattern in patterns:
            match = pattern.match(stem)
            if match and match.group(1):
                counts[match.group(1).rstrip("_-")] += 1
                break
    return dict(counts.most_common(20))


### Build an audit report

Collect annotation facts first. The following policy check applies the recorded experiment requirements separately.


In [ ]:
def audit_dataset(dataset: Path) -> dict:
    """Inspect annotation structure without writing files or enforcing split policy."""
    image_dir, label_dir = dataset / 'images', dataset / 'labels'
    failures = [f'Missing directory: {path}' for path in (image_dir, label_dir) if not path.is_dir()]
    images = image_paths(image_dir) if image_dir.is_dir() else []
    labels = sorted(label_dir.glob('*.txt')) if label_dir.is_dir() else []
    missing, malformed, counts = [], [], Counter()
    for image in images:
        label = label_dir / f'{image.stem}.txt'
        if not label.is_file():
            missing.append(image.stem)
            continue
        classes, errors = parse_label(label)
        counts.update(classes)
        malformed.extend(errors)
    extras = sorted({p.stem for p in labels} - {p.stem for p in images})
    if not images:
        failures.append('No images found')
    if missing:
        failures.append(f'{len(missing)} images lack labels')
    if malformed:
        failures.append(f'{len(malformed)} malformed annotation rows')
    if not counts:
        failures.append('No bounding boxes found')
    if counts and 0 not in counts:
        failures.append('Expected tobacco class 0 is absent')
    return dict(dataset=str(dataset), expected_class_zero='tobacco', images=len(images),
        labels=len(labels), boxes=sum(counts.values()), class_counts=dict(sorted(counts.items())),
        missing_labels=missing[:100], missing_labels_count=len(missing),
        extra_label_files=extras[:100], extra_label_files_count=len(extras),
        malformed_rows=malformed[:100], malformed_rows_count=len(malformed),
        likely_frame_prefixes=infer_prefixes([p.stem for p in images]),
        group_regex=None, group_regex_matches={}, status='fail' if failures else 'ok', failures=failures)


### Inspect a synthetic audit

This miniature fixture demonstrates the report structure. It intentionally does not meet the original dataset counts; the separate reproduction policy would reject it.


In [ ]:
with tempfile.TemporaryDirectory(prefix='tobacco-audit-example-') as example_dir:
    example_dataset = Path(example_dir)
    (example_dataset / 'images').mkdir()
    (example_dataset / 'labels').mkdir()
    # The annotation audit checks filenames and labels, not image decoding.
    (example_dataset / 'images/example.jpg').touch()
    (example_dataset / 'labels/example.txt').write_text('0 0.5 0.5 0.2 0.2\n', encoding='utf-8')
    example_report = audit_dataset(example_dataset)
display(pd.Series({key: example_report[key] for key in ['images', 'labels', 'boxes', 'class_counts', 'status']}))


## Preserve the original split

The saved manifests contain Windows paths. We read their filenames portably, verify them against `items.csv`, and later write absolute paths for this machine. We **do not generate a new random split**. Expected membership is 78 train, 16 validation, 16 test.

The split seed `20260627` controls dataset splitting; it is not a recorded training seed. The original runner relies on framework defaults for training randomness and augmentations. Its full resolved training arguments are not included here. Capturing the new environment makes differences auditable, but exact numerical replication is not guaranteed.

The saved manifest digest includes path strings and box counts, not image bytes. It is not a dataset-content checksum. Counts and filenames cannot establish that two datasets are identical; obtain archive checksums from the dataset owner when available. The split is image-based and does not establish independence by field or acquisition session.


In [ ]:
assignments = frozen_assignments(ROOT)
display(pd.DataFrame({'split': assignments.keys(), 'images': [len(v) for v in assignments.values()]}))


In [ ]:
def validate_reproduction(report: dict, dataset: Path, items, assignments: dict) -> None:
    """Require the recorded counts and exact frozen filenames before training."""
    if report['status'] != 'ok':
        raise ValueError('Dataset audit failed: ' + '; '.join(report['failures']))
    for key, expected in [('images', 110), ('boxes', 4436), ('extra_label_files_count', 0)]:
        if report[key] != expected:
            raise ValueError(f"{dataset}: expected {key}={expected}, found {report[key]}")
    if set(report['class_counts']) != {0}:
        raise ValueError(f"Expected only class 0; found {report['class_counts']}")
    expected_names = {name for names in assignments.values() for name in names}
    actual_names = {p.name for p in image_paths(dataset / 'images')}
    if expected_names != actual_names:
        raise ValueError(f'Frozen image membership differs: missing={sorted(expected_names - actual_names)}, '
                         f'extra={sorted(actual_names - expected_names)}')
    for item in items.itertuples():
        label = dataset / 'labels' / f'{item.stem}.txt'
        count = sum(bool(line.strip()) for line in label.read_text().splitlines())
        if count != item.boxes:
            raise ValueError(f'{label}: expected {item.boxes} boxes, found {count}')


## Validate inputs and prepare a fresh run

A mismatch stops reproduction. Inspect the report before training. Environment capture is infrastructure; dataset checks are visible above.


In [ ]:
training_complete = False
run_dir = None
if RUN_TRAINING:
    if not CLASS_ZERO_CONFIRMED_TOBACCO:
        raise ValueError('Confirm class 0 means tobacco before enabling training.')
    DATASET = DATASET.expanduser().resolve()
    preflight = audit_dataset(DATASET)
    display(pd.Series(preflight))
    items = pd.read_csv(ROOT / 'experiments/splits/items.csv')
    validate_reproduction(preflight, DATASET, items, frozen_assignments(ROOT))
    import torch
    if TRAINING_OPTIONS['device'] != 'cpu' and not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable: install compatible PyTorch/drivers or explicitly select cpu.')
    run_dir = new_run(ROOT, RUN_NAME)
    (run_dir / 'dataset_preflight.json').write_text(json.dumps(preflight, indent=2) + '\n', encoding='utf-8')
    local_splits = write_local_splits(ROOT, DATASET, run_dir / 'splits')
    capture_environment(run_dir / 'environment')
else:
    print('External dataset validation disabled. No run directory created.')


## Describe the data and collect artifacts

The generated YAML points to relocated manifests. Preserve training-log metrics separately from held-out evaluation metrics.


### Write the relocated dataset description


In [ ]:
def write_data_yaml(dataset, splits, output, class_name="tobacco") -> Path:
    yaml_path = output / "data.generated.yaml"
    yaml_path.write_text(
        "\n".join(
            [
                f"path: {json.dumps(str(dataset.resolve()))}",
                f"train: {json.dumps(str((splits / 'train.txt').resolve()))}",
                f"val: {json.dumps(str((splits / 'val.txt').resolve()))}",
                f"test: {json.dumps(str((splits / 'test.txt').resolve()))}",
                "names:",
                f"  0: {class_name}",
                "",
            ]
        ),
        encoding="utf-8",
    )
    return yaml_path


### Read the final training-log row


In [ ]:
def collect_metrics(run_dir: Path) -> dict:
    results_csv = run_dir / "results.csv"
    if not results_csv.exists():
        return {}
    with results_csv.open("r", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        return {}
    return {key.strip(): float(value) for key, value in rows[-1].items() if value.strip()}


### Extract held-out measurements


In [ ]:
def flatten_metrics(prefix: str, metrics: object) -> dict:
    output: dict = {}
    results_dict = getattr(metrics, "results_dict", None)
    if isinstance(results_dict, dict):
        for key, value in results_dict.items():
            output[f"{prefix}_{key}".replace(" ", "_")] = value
    speed = getattr(metrics, "speed", None)
    if isinstance(speed, dict):
        for key, value in speed.items():
            output[f"{prefix}_speed_{key}_ms"] = value
    return output


### Inspect checkpoint size


In [ ]:
def model_file_size_mb(path: Path) -> float | str:
    if not path.exists():
        return ""
    return path.stat().st_size / (1024 * 1024)


### Count model parameters


In [ ]:
def parameter_count(detector: object) -> int | str:
    model = getattr(detector, "model", None)
    if model is None:
        return ""
    return sum(parameter.numel() for parameter in model.parameters())


### Copy diagnostic artifacts


In [ ]:
def copy_artifacts(run_dir: Path, output: Path, model_name: str) -> None:
    for filename in ["results.csv", "results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
        source = run_dir / filename
        if source.exists():
            shutil.copy2(source, output / f"{model_name}_{filename}")


## Train and evaluate the selected checkpoints

The calls below expose the actual SGD settings and evaluation protocol. Each model records failures and the final completeness check rejects partial results. Selection uses best.pt, falling back to last.pt if best.pt is absent; both paths are recorded. Test results must not guide hyperparameter tuning.


In [ ]:
def evaluate_checkpoint(best_weights: Path, last_weights: Path, data_yaml: Path,
                        output: Path, run_name: str, model_factory, *, imgsz: int,
                        batch: int, device: str) -> dict:
    """Return independent validation/test statuses and numeric measurements."""
    checkpoint = best_weights if best_weights.is_file() else last_weights
    row = {}
    try:
        if not checkpoint.is_file():
            raise FileNotFoundError(f'Neither checkpoint exists: {best_weights}, {last_weights}')
        detector = model_factory(str(checkpoint))
        row['trained_parameters'] = parameter_count(detector)
    except Exception as exc:
        for split in ('val', 'test'):
            row.update({f'{split}_status': 'failed', f'{split}_error': f'{type(exc).__name__}: {exc}'})
        return row
    for split in ('val', 'test'):
        try:
            metrics = detector.val(data=str(data_yaml), split=split, imgsz=imgsz, batch=batch,
                device=device, plots=(split == 'val'), project=str(output.resolve()),
                name=f'{run_name}_{split}', exist_ok=False)
            row.update(flatten_metrics(split, metrics))
            if row.get(f'{split}_metrics/mAP50-95(B)') is None:
                raise ValueError('Evaluation did not return required mAP50-95 metric')
            row[f'{split}_status'] = 'ok'
        except Exception as exc:
            row[f'{split}_status'] = 'failed'
            row[f'{split}_error'] = f'{type(exc).__name__}: {exc}'
    return row


### Train one model

The training call explicitly specifies the historical optimizer and schedule. After fitting, collect the final log row, select a checkpoint, and evaluate it. Failures remain visible in the comparison table.


In [ ]:
def train_one_model(model: str, data_yaml: Path, output: Path, model_factory, *, options: dict) -> dict:
    """Fit one model; retain separate training and held-out evaluation outcomes."""
    model_name = Path(model).stem
    run_dir = output / model_name
    row = dict(model=model, model_name=model_name, run_dir=str(run_dir),
        **{key: options[key] for key in ('epochs', 'imgsz', 'batch', 'device')},
        status='failed', train_status='started', val_status='not_run', test_status='not_run', notes='')
    start = time.time()
    try:
        detector = model_factory(model)
        row['initial_parameters'] = parameter_count(detector)
        metrics = detector.train(data=str(data_yaml), **options, plots=True,
                                 project=str(output.resolve()), name=model_name, exist_ok=False)
        row.update(flatten_metrics('train_return', metrics))
        row.update(collect_metrics(run_dir))
        row['train_status'] = 'ok'
    except Exception as exc:
        row['train_status'] = 'failed'
        row['notes'] = f'{type(exc).__name__}: {exc}'
    row['train_seconds'] = time.time() - start
    if row['train_status'] == 'ok':
        best, last = run_dir / 'weights/best.pt', run_dir / 'weights/last.pt'
        row.update(best_weights=str(best) if best.is_file() else '',
                   last_weights=str(last) if last.is_file() else '', best_weights_mb=model_file_size_mb(best))
        row.update(evaluate_checkpoint(best, last, data_yaml, output, model_name, model_factory,
            imgsz=options['imgsz'], batch=options['batch'], device=options['device']))
        if all(row[f'{stage}_status'] == 'ok' for stage in ('train', 'val', 'test')):
            row['status'] = 'ok'
    copy_artifacts(run_dir, output, model_name)
    return row


### Compare models and save the experiment record

Apply the same settings to every model, export comparison rows, and record the run configuration. A subsequent completeness check rejects failed or missing evaluations.


In [ ]:
def train_detectors(dataset: Path, splits: Path, models: list[str], output: Path, *,
                    options: dict, model_factory=None) -> list[dict]:
    """Reserve a fresh output, compare models, and export exact training settings."""
    names = [Path(model).stem for model in models]
    if not models or len(set(names)) != len(names):
        raise ValueError('Supply models with unique nonempty run names.')
    output.mkdir(parents=True, exist_ok=False)
    Path(os.environ['YOLO_CONFIG_DIR']).mkdir(parents=True, exist_ok=True)
    if model_factory is None:
        from ultralytics import YOLO
        model_factory = YOLO
    data_yaml = write_data_yaml(dataset, splits, output)
    rows = [train_one_model(model, data_yaml, output, model_factory, options=options) for model in models]
    fieldnames = sorted({key for row in rows for key in row})
    with (output / 'model_comparison.csv').open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    metadata = dict(dataset=str(dataset), splits=str(splits), models=models, **options,
                    model_status={row['model_name']: row['status'] for row in rows})
    (output / 'run_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n', encoding='utf-8')
    return rows


## Run the comparison

For GPU memory errors, reduce batch size consistently for every model and choose a fresh run name. Report changes and CPU timing explicitly. Restart the kernel before retrying a failed experiment.


In [ ]:
training_complete = False
if RUN_TRAINING:
    if run_dir is None:
        raise RuntimeError('Run input validation and initialization first.')
    comparison_rows = train_detectors(DATASET, local_splits, MODELS, run_dir / 'yolo', options=TRAINING_OPTIONS)
    display(pd.DataFrame(comparison_rows))
    check_detector_results(run_dir / 'yolo/model_comparison.csv')
    training_complete = True
else:
    print('Training skipped; no new measurements generated.')


## Compare the new measurements with the historical run

Differences can arise from framework versions, randomness, GPU kernels, and hardware. Do not substitute historical timing for the new machine's timing. Report changed settings together with the captured environment. Test data should not be used to tune hyperparameters.


In [ ]:
if training_complete:
    old = pd.read_csv(RESULTS / 'yolo/model_comparison.csv').set_index('model_name')
    new = pd.read_csv(run_dir / 'yolo/model_comparison.csv').set_index('model_name')
    keys = ['test_metrics/precision(B)', 'test_metrics/recall(B)',
            'test_metrics/mAP50(B)', 'test_metrics/mAP50-95(B)', 'test_speed_inference_ms']
    display(pd.concat({'Historical': old[keys], 'Reproduction': new[keys]}, axis=1))
    print('Outputs:', run_dir)
    print('Use this trained model in notebook 3:', run_dir / 'yolo/yolov8x/weights/best.pt')
else:
    print('No new experiment was run; there are no new metrics to compare.')
